# Azure MLOps with AutoML Models: Advanced Notebook

You trained two classifiers with **Automated ML** in the `ml_5350` workspace and registered them as MLflow models:

| Registered model | Trained on | Expected behaviour (from the local lab) |
|---|---|---|
| `balanced_model` (v1) | `balanced.csv`: 1,000 ham + 1,000 spam | Catches spam well |
| `sms_real_1k` (v1) | `sms_real_1k.csv`: 85% ham | Risk of the accuracy trap: high accuracy, low spam recall |

This notebook takes those two models through a production MLOps loop **on Azure Machine Learning**, using the Python SDK v2 (`azure-ai-ml`) and MLflow. It is the cloud version of `mlops_pipeline.ipynb`: the same ideas (registry, quality gate, safe rollout, drift, retraining), done with the managed services a real team would use.

It follows the Microsoft Learn path **[Introduction to machine learning operations (MLOps)](https://learn.microsoft.com/en-us/training/paths/introduction-machine-learn-operations/)**:

| MS Learn module | Where in this notebook |
|---|---|
| Introduction to DevOps principles for machine learning | Sections 0, 7 and 10: architecture, environments, safe rollout |
| Source control for machine learning projects | Section 9: repo layout, unit tests, pull requests |
| Automate machine learning workflows | Sections 3, 4 and 8: Azure ML pipelines, AutoML jobs, GitHub Actions |
| Continuous deployment for machine learning | Sections 5, 6 and 9: managed endpoints, blue/green, approvals |

### The architecture you will build

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    REG["Model registry<br/>balanced_model, sms_real_1k"] --> PIPE["Azure ML pipeline<br/>evaluate + gate"]
    DATA["Data assets<br/>versioned CSVs"] --> PIPE
    PIPE --> TAGS["Registry tags<br/>gate = pass / fail"]
    TAGS --> EP["Managed online endpoint"]
    EP --> BLUE["blue<br/>champion"]
    EP --> GREEN["green<br/>challenger"]
    EP --> MON["Monitoring<br/>drift, logged to MLflow"]
    MON -- "drift" --> AML["AutoML retraining job"]
    AML --> REG
    GH["GitHub Actions<br/>tests, gate, approval"] -.-> PIPE
    GH -.-> EP
```

### Before you start

| You need | How to check |
|---|---|
| The `ml_5350` workspace with both models registered | Azure ML studio → **Models** |
| A role that can create jobs, endpoints and compute (Contributor or AzureML Data Scientist) | Ask your instructor |
| Azure CLI login, or a browser for interactive login | `az login --tenant <your-tenant-id>` |
| Python packages | `pip install -r requirements.txt` (in this folder) |
| VM quota for `Standard_DS3_v2` (4 cores each; up to 3 instances at once) | Studio → **Quotas** |

> ⚠️ **This notebook creates billable resources.** A compute cluster (scales to zero when idle), a managed online endpoint with up to two deployments (**billed every hour they exist, even with no traffic**), and an optional AutoML job. Run **section 11 (Clean up)** when you finish. Cells that create something billable are marked 💲.

## 0 · Connect to the workspace

Tell the notebook which workspace to use. Pick **one** way:

| Option | How | Best for |
|---|---|---|
| **`.env` file** (recommended) | `cp env.sample .env`, then fill in `SUBSCRIPTION_ID` and `RESOURCE_GROUP` | Keeping your IDs out of the notebook and out of git (`.env` is git-ignored) |
| `config.json` | Studio → top-right menu → *Download config file*, saved in this folder | One-click setup from the studio |
| This cell | Type the values below | A quick test |

If more than one is set, `config.json` wins, then `.env`, then this cell.

The other settings name the two models and the resources this notebook creates.

In [ ]:
import os

SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP = "<your-resource-group>"
WORKSPACE = "ml_5350"


def read_env_file(path=".env"):
    """Read KEY=VALUE lines. Spaces around '=' and quotes are fine; lines starting with # are comments."""
    values = {}
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_file = read_env_file()
SUBSCRIPTION_ID = env_file.get("SUBSCRIPTION_ID", SUBSCRIPTION_ID)
RESOURCE_GROUP = env_file.get("RESOURCE_GROUP", RESOURCE_GROUP)
WORKSPACE = env_file.get("WORKSPACE", WORKSPACE)
print(f"Settings from {'.env' if env_file else 'this cell'}: subscription {SUBSCRIPTION_ID[:8]}..., "
      f"resource group {RESOURCE_GROUP}, workspace {WORKSPACE}")

CHAMPION = {"name": "balanced_model", "version": "1"}   # current best model
CHALLENGER = {"name": "sms_real_1k", "version": "1"}    # the model that wants to replace it

COMPUTE = "cpu-cluster"            # created in section 2 if it does not exist
COMPUTE_SIZE = "Standard_DS3_v2"
INSTANCE_TYPE = "Standard_DS3_v2"  # for the online endpoint
ENDPOINT = None                    # None = generate a unique name (endpoint names are unique per region)

FEATURES = ["length", "punct"]
LABEL = "label"
DATA_DIR = "../data"               # the lab's CSVs

In [ ]:
import glob
import hashlib
import json
import os
import re
import string
import uuid

import mlflow
import numpy as np
import pandas as pd
from azure.ai.ml import Input, MLClient, automl, load_job
from azure.ai.ml.automl import ClassificationPrimaryMetrics
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import (AmlCompute, Data, Environment, ManagedOnlineDeployment,
                                  ManagedOnlineEndpoint, Model)
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from mlflow.tracking import MlflowClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()     # opens a browser window to sign in

if os.path.exists("config.json"):
    ml_client = MLClient.from_config(credential=credential, path="config.json")
else:
    ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)

workspace = ml_client.workspaces.get(ml_client.workspace_name)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)   # MLflow now reads and writes the workspace
mlflow_client = MlflowClient()

ENDPOINT = ENDPOINT or f"sms-spam-{uuid.uuid4().hex[:6]}"
print(f"Connected to {workspace.name} ({workspace.location}). Endpoint name: {ENDPOINT}")

`MLClient` manages Azure resources (jobs, models, endpoints). **MLflow** reads and writes runs, metrics and model tags in the same workspace, with the standard open-source API. In Azure ML you use both, side by side.

## 1 · The model registry: what do we have, and where did it come from?

*Slide 4: accountable.* AutoML registered each model with a link back to the **job** (run) that trained it. From that job, MLflow gives us the training metrics and the parent AutoML run, and the parent tells us which data it used. That chain is the model's **lineage**.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    M["registered model<br/>balanced_model:1"] --> J["child run<br/>balanced_job_2"]
    J --> P["AutoML parent job"]
    P --> D["training data asset"]
    J --> MET["metrics<br/>accuracy, AUC, recall"]
```

In [ ]:
rows = []
for spec in (CHAMPION, CHALLENGER):
    for m in ml_client.models.list(name=spec["name"]):
        rows.append({
            "name": m.name, "version": m.version, "type": m.type, "job (run id)": m.job_name,
            "created": m.creation_context.created_at.strftime("%Y-%m-%d %H:%M") if m.creation_context else None,
            "tags": m.tags,
        })
pd.DataFrame(rows)

### 1b · Lineage and AutoML's own metrics

AutoML logs many metrics. Its classification metrics are **macro** or **weighted** averages over both classes, not "recall for spam". Keep that in mind: a macro recall of 0.5 can hide a spam recall near 0.

In [ ]:
AUTOML_METRICS = ["accuracy", "balanced_accuracy", "AUC_weighted", "precision_score_macro",
                  "recall_score_macro", "f1_score_macro", "norm_macro_recall"]


def lineage(spec):
    model = ml_client.models.get(spec["name"], version=spec["version"])
    run = mlflow.get_run(model.job_name)                      # the AutoML child run that produced the model
    parent_id = run.data.tags.get("mlflow.parentRunId")
    training_data = None
    if parent_id:
        try:
            training_data = ml_client.jobs.get(parent_id).training_data.path
        except Exception as exc:
            training_data = f"(could not read parent job: {exc.__class__.__name__})"
    algorithm = "(not available)"
    try:   # stored on the child job, not in MLflow; some AutoML child runs can't be read through this API
        properties = ml_client.jobs.get(model.job_name).properties or {}
        algorithm = properties.get("run_algorithm", algorithm)
        if properties.get("ensembled_algorithms"):
            algorithm += f" of {properties['ensembled_algorithms']}"
    except Exception:
        pass
    return {
        "model": f"{model.name}:{model.version}",
        "trained by run": model.job_name,
        "AutoML parent job": parent_id,
        "training data": training_data,
        "algorithm": algorithm,
        **{k: round(v, 3) for k, v in run.data.metrics.items() if k in AUTOML_METRICS},
    }


lineage_table = pd.DataFrame([lineage(CHAMPION), lineage(CHALLENGER)]).set_index("model").T
lineage_table

**Q1.** Compare `accuracy` and `norm_macro_recall` for the two models. Which one does AutoML's *own* report already flag as weak at finding spam? Why is `norm_macro_recall` a better primary metric than `accuracy` for imbalanced data?

### 1c · What does the model expect? Read its MLflow signature

An MLflow model carries a **signature**: the input columns and types it accepts. Deployment and scoring rely on it. We download both models once; we will also reuse the files in section 2.

In [ ]:
def download_model(spec):
    target = f"downloads/{spec['name']}_{spec['version']}"
    if not glob.glob(f"{target}/**/MLmodel", recursive=True):
        ml_client.models.download(spec["name"], version=spec["version"], download_path=target)
    return os.path.dirname(glob.glob(f"{target}/**/MLmodel", recursive=True)[0])


model_dirs = {spec["name"]: download_model(spec) for spec in (CHAMPION, CHALLENGER)}
for name, folder in model_dirs.items():
    info = mlflow.models.Model.load(os.path.join(folder, "MLmodel"))
    print(f"{name}\n  inputs : {info.signature.inputs if info.signature else 'no signature'}"
          f"\n  outputs: {info.signature.outputs if info.signature else '-'}\n  files  : {sorted(os.listdir(folder))}\n")

The inputs should be `length` and `punct`, the two features from the lab. If your AutoML job was trained on a CSV with other column names, change `FEATURES` and `LABEL` in section 0 to match.

## 2 · Versioned data, a reproducible environment, and compute

*Slide 4: reproducible.* Three things must be pinned before any evaluation can be trusted: **the data** (versioned data assets), **the software** (an environment), and **where it runs** (compute).

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    CSV["lab CSVs"] -- "register,<br/>version = fingerprint" --> DA["data assets"]
    CONDA["model's conda.yaml"] -- "register" --> ENV["sms-automl-env"]
    CL["cpu-cluster<br/>min 0 nodes"] --> JOBS["evaluation jobs"]
    DA --> JOBS
    ENV --> JOBS
```

### 2a · Data assets, versioned by content

We register the fixed test set and the weekly batches as **data assets**. The version is a fingerprint of the file, so re-running this cell never creates duplicates, and changing a file automatically creates a new version.

In [ ]:
def file_fingerprint(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


def register_file(name, path, description):
    version = file_fingerprint(path)
    try:
        return ml_client.data.get(name, version=version)
    except ResourceNotFoundError:
        return ml_client.data.create_or_update(Data(
            name=name, version=version, path=path, type=AssetTypes.URI_FILE, description=description))


assets = {
    "sms_test_set": register_file("sms_test_set", f"{DATA_DIR}/test_set.csv", "Fixed test set: 500 messages, 15% spam"),
    "sms_week_ok": register_file("sms_week_ok", f"{DATA_DIR}/new_week_ok.csv", "Weekly batch: normal"),
    "sms_week_drift": register_file("sms_week_drift", f"{DATA_DIR}/new_week_drift.csv", "Weekly batch: spammers send short messages"),
}
pd.DataFrame([{"asset": a.name, "version": a.version, "path": a.path} for a in assets.values()])

### 2b · The environment: use the model's own dependencies

An AutoML model only loads with the exact libraries it was trained with. Those are listed in the `conda.yaml` that MLflow saved next to the model. We register that file as an Azure ML **environment**, `sms-automl-env`, so every evaluation job loads the model the same way. The first job that uses it builds a Docker image. AutoML's dependencies are large (Python 3.10, scikit-learn 1.5, LightGBM, XGBoost and even PyTorch), so expect **20–30 minutes** the first time. Later jobs reuse the image.

In [ ]:
conda_file = os.path.join(model_dirs[CHAMPION["name"]], "conda.yaml")
print(open(conda_file).read()[:600], "...")

env = ml_client.environments.create_or_update(Environment(
    name="sms-automl-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file=conda_file,
    description=f"Dependencies of {CHAMPION['name']}:{CHAMPION['version']} (AutoML MLflow model)",
))
print(f"\nenvironment {env.name}:{env.version}")

Both models were trained by AutoML in the same workspace, so they normally share dependencies. (In `ml_5350` the two `conda.yaml` files are identical.) If the challenger's evaluation step fails with an import error, register a second environment from the challenger's `conda.yaml` and point its step at it.

### 2c · 💲 A compute cluster that scales to zero

`min_instances=0` means you only pay while jobs run. Idle nodes shut down after 15 minutes.

AutoML jobs started from the studio run on **serverless** compute by default, so your workspace may not have a cluster yet. Pipelines can also run serverless (set `COMPUTE = "serverless"` and skip this cell), but a named cluster makes the cost easy to see and cap.

In [ ]:
try:
    cluster = ml_client.compute.get(COMPUTE)
    print(f"Using existing cluster {COMPUTE} ({cluster.size})")
except ResourceNotFoundError:
    cluster = ml_client.compute.begin_create_or_update(AmlCompute(
        name=COMPUTE, size=COMPUTE_SIZE, min_instances=0, max_instances=2, idle_time_before_scale_down=900,
    )).result()
    print(f"Created cluster {COMPUTE} ({COMPUTE_SIZE}, 0-2 nodes)")

## 3 · 💲 Evaluate and gate, as an Azure ML pipeline

*Slides 7 and 11. MS Learn: Automate machine learning workflows.* We do **not** trust AutoML's own validation scores for promotion decisions. Each AutoML run used its own split, so the numbers are not comparable. Instead, a **pipeline** scores both models on the **same fixed test set** used in the local lab, then a **gate** step decides.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    CH["champion<br/>balanced_model:1"] --> E1["evaluate_champion"]
    CL["challenger<br/>sms_real_1k:1"] --> E2["evaluate_challenger"]
    T["sms_test_set"] --> E1
    T --> E2
    E1 --> G["gate<br/>accuracy ≥ 0.85<br/>recall ≥ 0.80<br/>no regression"]
    E2 --> G
    G --> D["decision.json"]
```

The pipeline is defined in YAML (`pipelines/evaluate-and-gate.yml`), and its steps are reusable **components** (`components/*.yml`) that run `src/evaluate.py` and `src/gate.py`. The *same* files are used by GitHub Actions in section 9. One definition, no copy-paste drift between the notebook and CI.

In [ ]:
print(open("src/gate.py").read().split("def read_metrics")[0])

In [ ]:
pipeline_job = load_job("pipelines/evaluate-and-gate.yml", params_override=[
    {"inputs.champion_model.path": f"azureml:{CHAMPION['name']}:{CHAMPION['version']}"},
    {"inputs.challenger_model.path": f"azureml:{CHALLENGER['name']}:{CHALLENGER['version']}"},
    {"inputs.test_data.path": f"azureml:sms_test_set:{assets['sms_test_set'].version}"},
    {"settings.default_compute": f"azureml:{COMPUTE}"},
])
submitted = ml_client.jobs.create_or_update(pipeline_job)
print("Pipeline submitted. Watch it in the studio:", submitted.studio_url)
ml_client.jobs.stream(submitted.name)     # waits; the first run also builds the environment image

In [ ]:
def read_decision(pipeline_name, folder):
    ml_client.jobs.download(pipeline_name, output_name="decision", download_path=folder)
    with open(glob.glob(f"{folder}/**/decision.json", recursive=True)[0]) as f:
        return json.load(f)


decision = read_decision(submitted.name, "outputs/gate_v1")
compare = pd.DataFrame({
    f"champion {CHAMPION['name']}": decision["champion"],
    f"challenger {CHALLENGER['name']}": decision["challenger"],
}).loc[["accuracy", "precision", "recall", "f1", "predicted_positive_share"]]
print("PROMOTE challenger:", decision["promote"])
for reason in decision["reasons"]:
    print("  -", reason)
compare

**Q2.** Does the challenger pass? Compare its accuracy with its spam recall. Then compare these numbers with AutoML's own metrics from section 1b: why can the two disagree?

## 4 · Record the decision in the registry

Azure ML's MLflow registry does not have "Staging/Production" stages. The convention is **tags**. We tag each model version with the gate result, the test set version it was judged on and the pipeline run, so anyone browsing **Models** in the studio can see why a model is (or is not) in production.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    D["decision.json"] --> T1["champion tags<br/>stage = production"]
    D --> T2["challenger tags<br/>gate = pass or fail"]
```

In [ ]:
def tag_model(spec, **tags):
    for key, value in tags.items():
        mlflow_client.set_model_version_tag(spec["name"], spec["version"], key, str(value))


common = {"evaluated_on": f"sms_test_set:{assets['sms_test_set'].version}", "gate_run": submitted.name}
tag_model(CHAMPION, stage="production", **common,
          spam_recall=decision["champion"]["recall"], accuracy_fixed_test=decision["champion"]["accuracy"])
tag_model(CHALLENGER, stage="candidate", gate="pass" if decision["promote"] else "fail", **common,
          spam_recall=decision["challenger"]["recall"], accuracy_fixed_test=decision["challenger"]["accuracy"])

for spec in (CHAMPION, CHALLENGER):
    print(spec["name"], ml_client.models.get(spec["name"], version=spec["version"]).tags)

## 5 · 💲 Deploy the champion to a managed online endpoint

*Slides 9 and 10. MS Learn: Continuous deployment.* A **managed online endpoint** is a stable HTTPS address. Behind it you can run several **deployments** (model versions) and decide how traffic is split between them. Azure manages the VMs, scaling, TLS and logs.

Because the models are MLflow models, this is a **no-code deployment**: Azure ML generates the scoring script and the serving environment from the MLflow files.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    APP["app"] -- "HTTPS + key" --> EP["endpoint<br/>stable URL"]
    EP -- "100%" --> BLUE["blue<br/>balanced_model:1"]
    EP -. "0%" .-> GREEN["green<br/>(later)"]
```

Creating the endpoint takes about 1–2 minutes, and the deployment about 8–15 minutes.

In [ ]:
endpoint = ml_client.online_endpoints.begin_create_or_update(ManagedOnlineEndpoint(
    name=ENDPOINT, auth_mode="key", description="SMS spam classifier (MLOps 101)",
    tags={"course": "5350", "lab": "mlops-101"},
)).result()
print("Endpoint:", endpoint.scoring_uri)

blue = ml_client.online_deployments.begin_create_or_update(ManagedOnlineDeployment(
    name="blue", endpoint_name=ENDPOINT,
    model=f"azureml:{CHAMPION['name']}:{CHAMPION['version']}",
    instance_type=INSTANCE_TYPE, instance_count=1,
)).result()

endpoint = ml_client.online_endpoints.get(ENDPOINT)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic:", ml_client.online_endpoints.get(ENDPOINT).traffic)

### 5b · Call it

MLflow deployments expect a JSON body with `input_data` in pandas *split* format. `score()` below sends any DataFrame and returns the predicted labels. Pass `deployment=` to call one deployment directly, bypassing the traffic split.

In [ ]:
def score(df, deployment=None):
    body = {"input_data": {"columns": FEATURES, "index": list(range(len(df))),
                           "data": df[FEATURES].astype(int).values.tolist()}}
    with open("request.json", "w") as f:
        json.dump(body, f)
    raw = ml_client.online_endpoints.invoke(endpoint_name=ENDPOINT, request_file="request.json",
                                            deployment_name=deployment)
    result = json.loads(raw)
    if isinstance(result, dict):                    # some MLflow servers wrap the list
        result = result.get("predictions", next(iter(result.values())))
    return np.asarray(result).astype(str).ravel()


def features(text):
    """The same feature code as the lab (common.py): characters and punctuation marks."""
    return {"length": len(text), "punct": sum(1 for c in text if c in string.punctuation)}


examples = ["See you at lunch",
            "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize!!!",
            "WINNER!! Claim your free prize now!!!"]
pd.DataFrame([features(t) for t in examples]).assign(message=examples, prediction=score(pd.DataFrame([features(t) for t in examples])))

> The app must compute `length` and `punct` with *exactly* the same code as the training data did. If it counted punctuation even slightly differently, the model would get inputs it never saw in training, and nothing would raise an error. Keeping feature code identical between training and serving is the problem a **feature store** solves (`MLOPS_README.md`).

In [ ]:
test = pd.read_csv(f"{DATA_DIR}/test_set.csv")
live = score(test)
print(f"Live endpoint on the fixed test set: accuracy {np.mean(live == test[LABEL]):.1%}, "
      f"spam recall {np.mean(live[test[LABEL] == 'spam'] == 'spam'):.1%}")
print("\nLast lines of the blue deployment's log:")
print(ml_client.online_deployments.get_logs(name="blue", endpoint_name=ENDPOINT, lines=8))

**Q3.** The live endpoint's accuracy and recall should match the champion's numbers from the pipeline in section 3. Why is checking this worth the effort?

## 6 · 💲 Safe rollout with blue/green, shadow and canary

*Slide 12.* Azure ML endpoints implement all three strategies natively:

| Strategy | Endpoint setting | Users affected |
|---|---|---|
| **Shadow** | `mirror_traffic = {"green": 20}`: 20% of requests are *copied* to green, and its answers are discarded | None |
| **Canary** | `traffic = {"blue": 90, "green": 10}` | 10% |
| **Blue/green switch** | `traffic = {"green": 100}`, and back for a rollback | All, but undone in seconds |

We deploy the **challenger** to `green`, even though the gate failed it, to show that the system *refuses* to give it real traffic.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    U["requests"] --> EP{"endpoint"}
    EP -- "100% live" --> B["blue<br/>champion"]
    EP -. "20% copy" .-> G["green<br/>challenger"]
    G -. "answers discarded" .-> X["metrics only"]
```

In [ ]:
green = ml_client.online_deployments.begin_create_or_update(ManagedOnlineDeployment(
    name="green", endpoint_name=ENDPOINT,
    model=f"azureml:{CHALLENGER['name']}:{CHALLENGER['version']}",
    instance_type=INSTANCE_TYPE, instance_count=1,
)).result()

endpoint = ml_client.online_endpoints.get(ENDPOINT)
endpoint.traffic = {"blue": 100, "green": 0}
endpoint.mirror_traffic = {"green": 20}       # shadow: copy 20% of live requests to green
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
endpoint = ml_client.online_endpoints.get(ENDPOINT)
print("live traffic:", endpoint.traffic, "| mirrored:", endpoint.mirror_traffic)

### 6b · Compare blue and green on the same messages (no labels needed)

Mirrored responses are thrown away, so their only use is to watch green's latency and errors in **Azure Monitor** (endpoint → *Monitoring*). To compare *predictions*, we call both deployments directly with the same messages, as an offline shadow test.

In [ ]:
blue_pred = score(test, deployment="blue")
green_pred = score(test, deployment="green")
shadow = {
    "agreement": np.mean(blue_pred == green_pred),
    "blue says spam": np.mean(blue_pred == "spam"),
    "green says spam": np.mean(green_pred == "spam"),
}
{k: f"{v:.1%}" for k, v in shadow.items()}

### 6c · A traffic policy that enforces the gate

Anyone with access can type `traffic = {"green": 50}`. A safer team wraps traffic changes in a function that **checks the registry tag first**. That way, a model that failed the gate cannot receive live traffic, no matter who asks.

In [ ]:
def model_of(deployment_name):
    """name and version of the model behind a deployment, read from its ARM id"""
    model_id = ml_client.online_deployments.get(deployment_name, endpoint_name=ENDPOINT).model
    name, version = re.search(r"models/([^/]+)/versions/([^/]+)", model_id).groups()
    return {"name": name, "version": version}


def set_traffic(blue, green):
    if green > 0:
        spec = model_of("green")
        gate = ml_client.models.get(spec["name"], version=spec["version"]).tags.get("gate")
        if gate != "pass":
            print(f"REFUSED: {spec['name']}:{spec['version']} has gate = {gate!r}. It cannot get live traffic.")
            return False
    endpoint = ml_client.online_endpoints.get(ENDPOINT)
    endpoint.traffic = {"blue": blue, "green": green}
    endpoint.mirror_traffic = {} if green > 0 else endpoint.mirror_traffic   # mirroring and live traffic can't mix
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print("traffic now:", ml_client.online_endpoints.get(ENDPOINT).traffic)
    return True


set_traffic(blue=90, green=10)     # try a 10% canary for the challenger

### 6d · Remove the challenger

The gate failed, so `green` is deleted. Blue keeps 100% of the traffic the whole time: users never saw the challenger. (A **rollback** after a bad canary is the same move: `set_traffic(100, 0)`, then delete `green`.)

In [ ]:
endpoint = ml_client.online_endpoints.get(ENDPOINT)
endpoint.traffic, endpoint.mirror_traffic = {"blue": 100, "green": 0}, {}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
ml_client.online_deployments.begin_delete(name="green", endpoint_name=ENDPOINT).result()
print("deployments:", [d.name for d in ml_client.online_deployments.list(ENDPOINT)],
      "| traffic:", ml_client.online_endpoints.get(ENDPOINT).traffic)

**Q4.** In 6b, the challenger was compared with the champion *without labels*. What does a big gap in "says spam" tell you, and why is that useful before labels exist?

## 7 · Monitoring: watch for drift through the live endpoint

*Slide 13.* We send each weekly batch through the **live endpoint**, the way real traffic would arrive, and compare it with the training data. There are two label-free signals:
- **data drift:** the average `length` and `punct`
- **prediction drift:** the share of messages called spam

Each report is logged as an **MLflow run** in the `sms-monitoring` experiment, so the history is visible in the studio (*Jobs* → `sms-monitoring`) and can be charted over time.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    W["weekly batch"] --> EP["live endpoint"]
    EP --> R["drift report"]
    REF["training data"] --> R
    R --> ML["MLflow run<br/>sms-monitoring"]
    R -- "change > 25%" --> AL["ALERT"]
```

In [ ]:
reference = pd.read_csv(f"{DATA_DIR}/balanced.csv")
reference_spam_share = np.mean(score(reference) == "spam")
DRIFT_THRESHOLD = 0.25

mlflow.set_experiment("sms-monitoring")


def monitor(batch_file, name):
    batch = pd.read_csv(batch_file)
    predictions = score(batch)
    checks = {f"avg_{f}": (reference[f].mean(), batch[f].mean()) for f in FEATURES}
    checks["predicted_spam_share"] = (reference_spam_share, np.mean(predictions == "spam"))
    report = pd.DataFrame(checks, index=["reference", "this_week"]).T
    report["change"] = (report["this_week"] - report["reference"]) / report["reference"]
    report["drift"] = report["change"].abs() > DRIFT_THRESHOLD

    with mlflow.start_run(run_name=f"monitor-{name}"):
        mlflow.set_tags({"endpoint": ENDPOINT, "batch": name, "drift": str(bool(report["drift"].any()))})
        mlflow.log_metrics({f"{k}_change": float(v) for k, v in report["change"].items()})
        mlflow.log_metrics({f"{k}_this_week": float(v) for k, v in report["this_week"].items()})
        if LABEL in batch:     # labels arrived later: also log real performance
            mlflow.log_metrics({"recall_spam": float(np.mean(predictions[batch[LABEL] == "spam"] == "spam")),
                                "accuracy": float(np.mean(predictions == batch[LABEL]))})
        mlflow.log_dict(report.round(4).reset_index().to_dict(orient="records"), "drift_report.json")
    print(f"{name}: {'ALERT, drift in ' + ', '.join(report.index[report['drift']]) if report['drift'].any() else 'no drift'}")
    return report.round(3)


display(monitor(f"{DATA_DIR}/new_week_ok.csv", "week_ok"))
display(monitor(f"{DATA_DIR}/new_week_drift.csv", "week_drift"))

**Managed alternatives.** In production you would not run this by hand:
- **Azure Monitor** already tracks request rate, latency, errors and CPU for every deployment (endpoint → *Monitoring*). Set alert rules on them.
- **Azure ML model monitoring** compares collected production inputs and outputs with a baseline on a schedule, and raises alerts. It needs **data collection** turned on for the deployment. See [Monitor models in production](https://learn.microsoft.com/azure/machine-learning/concept-model-monitoring).
- An alert can trigger the **retraining** in section 8, through an Azure Logic App, an Event Grid subscription or a GitHub `repository_dispatch` event.

**Q5.** Which signal fired first in the drift week: data drift, prediction drift, or performance? In production, which of the three would you see first, and which last?

## 8 · 💲 Optional: retrain with an AutoML job, then send it through the same gate

*MS Learn: Automate machine learning workflows.* Here a new week of labeled data has arrived. We retrain with **AutoML from code**, using the same settings for every run (no clicking in the studio), and register the best model as a **new version** of the champion. Then we judge it with the **exact same pipeline** from section 3.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    NEW["balanced + new week"] --> MT["MLTable data asset"]
    MT --> AJ["AutoML job<br/>primary metric:<br/>norm_macro_recall"]
    AJ --> BEST["best child run"]
    BEST --> REG["register<br/>balanced_model:2"]
    REG --> GATE["evaluate-and-gate<br/>v1 vs v2"]
    GATE -- "pass" --> CAN["canary"]
    GATE -- "fail" --> KEEP["keep v1"]
```

> This cell is optional. An AutoML job with these limits usually takes 20–40 minutes on the 2-node cluster. Set `RUN_AUTOML = True` to run it.

In [ ]:
RUN_AUTOML = False

# AutoML reads tabular training data as an MLTable: a folder with the data and a small YAML file describing it
train_dir = "outputs/train_balanced_plus_week_ok"
os.makedirs(train_dir, exist_ok=True)
pd.concat([pd.read_csv(f"{DATA_DIR}/balanced.csv"), pd.read_csv(f"{DATA_DIR}/new_week_ok.csv")]) \
    .to_csv(f"{train_dir}/train.csv", index=False)
with open(f"{train_dir}/MLTable", "w") as f:
    f.write("paths:\n  - file: ./train.csv\ntransformations:\n  - read_delimited:\n"
            "      delimiter: ','\n      encoding: utf8\n      header: all_files_same_headers\n")

train_version = file_fingerprint(f"{train_dir}/train.csv")
try:
    train_asset = ml_client.data.get("sms_train_week_ok", version=train_version)
except ResourceNotFoundError:
    train_asset = ml_client.data.create_or_update(Data(
        name="sms_train_week_ok", version=train_version, path=train_dir, type=AssetTypes.MLTABLE,
        description="balanced.csv + new_week_ok.csv, for retraining"))
print("training data:", f"{train_asset.name}:{train_asset.version}")

In [ ]:
if RUN_AUTOML:
    automl_job = automl.classification(
        compute=COMPUTE,
        experiment_name="sms-retrain",
        training_data=Input(type=AssetTypes.MLTABLE, path=f"azureml:{train_asset.name}:{train_asset.version}"),
        target_column_name=LABEL,
        primary_metric=ClassificationPrimaryMetrics.NORM_MACRO_RECALL,   # not accuracy: see Q1
        n_cross_validations=5,
        enable_model_explainability=False,
    )
    automl_job.set_limits(timeout_minutes=40, trial_timeout_minutes=10, max_trials=6,
                          max_concurrent_trials=2, enable_early_termination=True)
    automl_job.set_training(allowed_training_algorithms=["LogisticRegression", "LightGBM", "RandomForest"])

    automl_run = ml_client.jobs.create_or_update(automl_job)
    print("AutoML job:", automl_run.studio_url)
    ml_client.jobs.stream(automl_run.name)

    best_run_id = mlflow.get_run(automl_run.name).data.tags["automl_best_child_run_id"]
    new_model = ml_client.models.create_or_update(Model(
        name=CHAMPION["name"],
        path=f"azureml://jobs/{best_run_id}/outputs/artifacts/paths/outputs/mlflow-model/",
        type=AssetTypes.MLFLOW_MODEL,
        description="AutoML retrain on balanced.csv + new_week_ok.csv",
        tags={"trained_on": f"{train_asset.name}:{train_asset.version}", "automl_job": automl_run.name},
    ))
    NEW_CHALLENGER = {"name": new_model.name, "version": new_model.version}
    print("registered", NEW_CHALLENGER)
else:
    print("Skipped. Set RUN_AUTOML = True to retrain.")

In [ ]:
if RUN_AUTOML:
    retrain_gate = ml_client.jobs.create_or_update(load_job("pipelines/evaluate-and-gate.yml", params_override=[
        {"inputs.champion_model.path": f"azureml:{CHAMPION['name']}:{CHAMPION['version']}"},
        {"inputs.challenger_model.path": f"azureml:{NEW_CHALLENGER['name']}:{NEW_CHALLENGER['version']}"},
        {"inputs.test_data.path": f"azureml:sms_test_set:{assets['sms_test_set'].version}"},
        {"settings.default_compute": f"azureml:{COMPUTE}"},
    ]))
    ml_client.jobs.stream(retrain_gate.name)
    new_decision = read_decision(retrain_gate.name, "outputs/gate_retrain")
    tag_model(NEW_CHALLENGER, stage="candidate", gate="pass" if new_decision["promote"] else "fail",
              evaluated_on=f"sms_test_set:{assets['sms_test_set'].version}", gate_run=retrain_gate.name)
    print("promote:", new_decision["promote"], new_decision["reasons"])

**Your turn (if the retrained model passed):** deploy `NEW_CHALLENGER` to `green` (copy the cell from 6a), run `set_traffic(90, 10)` (it is allowed this time, because the tag says `pass`), watch the endpoint metrics, then finish with `set_traffic(0, 100)`. When you are confident, delete `blue`. The new model is now the champion, so update `CHAMPION` and its `stage` tag.

**Q6.** The retraining job used `norm_macro_recall` as its primary metric and only three algorithms, with hard time limits. Give one reason for *each* of those three choices.

## 9 · CI/CD with GitHub Actions

*MS Learn: Source control, Automate workflows, Continuous deployment.* Everything you ran by hand is now a workflow that runs on a GitHub runner. It reuses the same YAML and scripts:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    T["test<br/>flake8 + pytest"] --> E["evaluate<br/>pipeline YAML"]
    E -- "promote = true" --> A["approval<br/>environment:<br/>production"]
    A --> D["deploy green<br/>+ canary %"]
    E -- "promote = false" --> S["stop"]
```

### How the repository is organised

```
azure-mlops/
├── azure_mlops.ipynb          # this notebook
├── src/evaluate.py            # pipeline step: score a model on a test set
├── src/gate.py                # pipeline step + library: the promotion rules
├── components/*.yml           # Azure ML components that wrap the two scripts
├── pipelines/evaluate-and-gate.yml
├── endpoints/*.yml            # endpoint and deployment definitions
├── tests/test_gate.py         # unit tests for the rules
└── ci/azure-mlops.yml         # GitHub Actions workflow (copy to .github/workflows/)
```

Good source-control habits from the MS Learn module apply directly:
- **Protect `main`.** Changes to the gate rules go through a pull request that a second person reviews, because changing `RECALL_MIN` changes what can reach users.
- **Test the rules.** `tests/test_gate.py` encodes the lab's lessons as tests: the accuracy trap must fail, and the bad week must be blocked. CI runs them on every change.
- **Code, not clicks.** The pipeline, endpoint and deployments are YAML in git, so every change is reviewed and versioned.
- **Environments.** The `deploy` job uses a GitHub *environment* called `production` with required reviewers: a human approves before any traffic moves. A larger setup adds a `staging` workspace first.

Run the unit tests here, exactly as CI does:

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True).stdout[-600:])

In [ ]:
print(open("ci/azure-mlops.yml").read())

**To turn it on:**
1. Copy `ci/azure-mlops.yml` to `.github/workflows/azure-mlops.yml` in your repository.
2. Create an identity GitHub can use. Register an app in Microsoft Entra ID, add a **federated credential** for your repo, and give it a role on the workspace's resource group. This is OpenID Connect, so no passwords are stored in GitHub. See [Use GitHub Actions with Azure Machine Learning](https://learn.microsoft.com/azure/machine-learning/how-to-github-actions-machine-learning).
3. Add secrets `AZURE_CLIENT_ID`, `AZURE_TENANT_ID` and `AZURE_SUBSCRIPTION_ID`, and variables `AZURE_RESOURCE_GROUP`, `AZURE_ML_WORKSPACE` and `AZURE_ML_ENDPOINT`.
4. Under *Settings → Environments*, create `production` with yourself as a required reviewer.
5. *Actions → azure-mlops → Run workflow*, with the model name and version to evaluate.

**Q7.** Why does the workflow only run on `workflow_dispatch` (by hand) instead of on every push? What event would you use in a mature setup?

## 10 · Where this puts you on the maturity ladder

*Slide 8.*

| Level | What it needs | In this notebook |
|---|---|---|
| 2 · Automated training | Repeatable, versioned training | AutoML job from code, versioned data assets, registered environment |
| 3 · Automated deployment | CI/CD ships the model | GitHub Actions: tests → gate pipeline → approval → canary |
| 4 · Full MLOps | Monitoring and automatic retraining | Drift reports in MLflow, plus a retraining job that reuses the gate |

What still separates this from a real Level 4 system: the drift alert does not yet *trigger* the retraining by itself, labels for new data are collected by hand, and there is a single workspace instead of separate dev, staging and production workspaces.

**Q8.** Name the single change that would most reduce risk in this setup, and the single change that would most reduce manual work.

## 11 · 💲 Clean up

The endpoint is the expensive part: each deployment instance is billed every hour it exists. Delete it when you finish. The cluster scales to zero by itself, but you can delete it too. Models, data assets, environments and MLflow runs cost almost nothing to keep, and they are your audit trail.

In [ ]:
DELETE_ENDPOINT = True
DELETE_CLUSTER = False

if DELETE_ENDPOINT:
    ml_client.online_endpoints.begin_delete(name=ENDPOINT).result()
    print("deleted endpoint", ENDPOINT)
if DELETE_CLUSTER:
    ml_client.compute.begin_delete(COMPUTE).result()
    print("deleted cluster", COMPUTE)

---
## Answers

**A1.** The model trained on `sms_real_1k.csv`. In `ml_5350` it has `accuracy` 0.857 but `norm_macro_recall` 0.426 and `balanced_accuracy` 0.713, while `balanced_model` scores 0.981, 0.963 and 0.981. `norm_macro_recall` rescales macro recall so that random guessing scores 0 and perfect recall on *every* class scores 1. A model that always says "ham" gets 85% accuracy but a `norm_macro_recall` of 0. It cannot hide behind the majority class. AutoML did better than the local lab's logistic regression (which caught 4% of spam), because it tried stronger algorithms, but the gap is still large. A balanced accuracy of 0.71 with ~97% of ham correct implies only about half the spam is caught.

**A2.** If `sms_real_1k` learned the majority class, it shows high accuracy (around 85%) but low spam recall, so the gate **blocks** it. That is the accuracy trap from slide 7, caught automatically. The numbers can differ from AutoML's because (1) AutoML scored on its own validation split or cross-validation folds, not on our fixed test set, and (2) AutoML reports macro or weighted averages, while the gate looks at spam recall specifically. Promotion decisions must use the same test set and the metric users care about.

**A3.** It proves that **what was evaluated is what is serving**. Many production bugs come from a gap between the two: a different model version, a different preprocessing step, or a different input format. Matching numbers on the same test set is a cheap end-to-end check. Good CD pipelines run it automatically right after every deployment, as a "smoke test".

**A4.** If green flags far fewer messages as spam than blue on the same traffic, at least one of them is wrong on many messages. Since blue passed the gate, green is the suspect. This is available **immediately** on real traffic, while labels may take days. It is the same signal that caught the bad model in the local lab's shadow test (1.8% vs 15.6%).

**A5.** In this notebook all three appear at once, because the drift batch already has labels. In production the order is: **data drift** and **prediction drift** are available the moment messages arrive (no labels needed), and **performance** (spam recall) only once someone labels a sample, often days or weeks later. That is why monitoring leans on the label-free signals as the early alarm.

**A6.** **Primary metric:** AutoML picks the "best" model by this metric, so it must match the job. `norm_macro_recall` rewards models that find spam, not ones that win on the majority class. **Few algorithms:** fewer trials means a faster, cheaper and more repeatable run. For two numeric features, more complex models add little. **Time limits and early termination:** a pipeline that runs every week must have a predictable cost and finish time. A job that can run for hours cannot be scheduled reliably.

**A7.** Each run starts cloud jobs and may change live traffic. Triggering that on every push of a typo fix would waste money and risk surprises while a team is still learning. In a mature setup, a **new model version** or **new data** starts the run instead: an Event Grid event from Azure ML when a model is registered, a schedule, or a `repository_dispatch` from the drift alert in section 7. Code changes to `src/` and `tests/` still run the `test` job on every pull request.

**A8.** **Risk:** a separate **staging** endpoint or workspace, where every new model serves real-shaped traffic (and passes the smoke test from Q3) before production. Today the gate is the only barrier. **Manual work:** connect the monitoring alert to the workflow (a `repository_dispatch` or schedule), and set up a **labeling loop** (for example, users' "report spam" clicks), so retraining data arrives without anyone exporting CSVs.